In [1]:
!pip install implicit pandas numpy scipy scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 56.3 MB/s eta 0:00:00


In [2]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

from tqdm import tqdm

In [3]:
try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42

DATA_ZIP = Path("submissions_archive_2026-05-14.zip")
DATA_DIR = Path("submissions_export")

OUTPUT_DIR = Path("hybrid_recommender_output_with_trend")
OUTPUT_DIR.mkdir(exist_ok=True)

Достаем данные

In [4]:
def read_export_csv(file_name):
    path = DATA_DIR / file_name

    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

    # sep=None сам определяет разделитель: ; или ,
    df = pd.read_csv(
        path,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )

    # Чистим названия колонок
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    return df

In [7]:
import zipfile
from pathlib import Path

zip_path = Path("submissions_archive_2026-05-14.zip")  # подставь точное имя архива
extract_dir = Path("submissions_export")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted files:")
for p in extract_dir.rglob("*"):
    print(p)

Extracted files:
submissions_export/README.txt
submissions_export/task_competence_mapping.csv
submissions_export/micromodules_competencies.csv
submissions_export/user_competences.csv
submissions_export/user_task_submissions.csv
submissions_export/user_practice_submissions.csv


In [8]:
user_comp_df = read_export_csv("user_competences.csv")
task_sub_df = read_export_csv("user_task_submissions.csv")
practice_df = read_export_csv("user_practice_submissions.csv")
task_comp_df = read_export_csv("task_competence_mapping.csv")
micro_df = read_export_csv("micromodules_competencies.csv")

In [9]:
def normalize_user_column(df):
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    # В новой выгрузке user_id называется user_number.
    # Для нашего кода создаём единое поле user_id.
    if "user_id" not in df.columns and "user_number" in df.columns:
        df["user_id"] = df["user_number"].astype(str)

    return df


user_comp_df = normalize_user_column(user_comp_df)
task_sub_df = normalize_user_column(task_sub_df)
practice_df = normalize_user_column(practice_df)
task_comp_df = normalize_user_column(task_comp_df)
micro_df = normalize_user_column(micro_df)

In [10]:
print(user_comp_df.shape)
print(user_comp_df.columns.tolist())
display(user_comp_df.head())

(7850, 11)
['user_number', 'competence_id', 'competence_code', 'competence_name', 'probability', 'assessment_source', 'evidence_count', 'assessed_at', 'valid_from', 'valid_to', 'user_id']


,user_number,competence_id,competence_code,competence_name,probability,assessment_source,evidence_count,assessed_at,valid_from,valid_to,user_id
0,1,e0bf6fb3-c7cf-47af-8f34-aa79a7837321,10.0.0,Текстовые задачи,0.999026,calculated,9,2026-02-25 17:41:52.810162,2026-02-25 17:41:52.810162,2026-02-25 17:42:18.335075,1
1,1,e0bf6fb3-c7cf-47af-8f34-aa79a7837321,10.0.0,Текстовые задачи,1.000000,calculated,18,2026-02-25 17:42:18.335075,2026-02-25 17:42:18.335075,2026-02-25 18:31:33.982309,1
2,1,e0bf6fb3-c7cf-47af-8f34-aa79a7837321,10.0.0,Текстовые задачи,1.000000,calculated,26,2026-02-25 18:31:33.982309,2026-02-25 18:31:33.982309,2026-03-15 17:47:24.225179,1
3,1,e0bf6fb3-c7cf-47af-8f34-aa79a7837321,10.0.0,Текстовые задачи,1.000000,calculated,27,2026-03-15 17:47:24.225179,2026-03-15 17:47:24.225179,2026-03-15 18:13:54.078542,1
4,1,e0bf6fb3-c7cf-47af-8f34-aa79a7837321,10.0.0,Текстовые задачи,1.000000,calculated,28,2026-03-15 18:13:54.078542,2026-03-15 18:13:54.078542,2026-03-16 18:15:52.994341,1


Берем последнее актуальное состояние user x comp

In [11]:
def prepare_user_competences(df):
    uc = df.copy()

    required_cols = ["user_id", "competence_id", "probability", "evidence_count"]
    missing = [col for col in required_cols if col not in uc.columns]
    if missing:
        raise ValueError(f"В user_competences не хватает колонок: {missing}")

    uc["user_id"] = uc["user_id"].astype(str)
    uc["competence_id"] = uc["competence_id"].astype(str)

    if "competence_code" in uc.columns:
        uc["competence_code"] = uc["competence_code"].astype(str)
    else:
        uc["competence_code"] = ""

    if "competence_name" in uc.columns:
        uc["competence_name"] = uc["competence_name"].astype(str)
    else:
        uc["competence_name"] = ""

    uc["probability"] = pd.to_numeric(uc["probability"], errors="coerce")
    uc["evidence_count"] = pd.to_numeric(uc["evidence_count"], errors="coerce").fillna(0)

    if "assessed_at" in uc.columns:
        uc["assessed_at"] = pd.to_datetime(uc["assessed_at"], errors="coerce")
    else:
        uc["assessed_at"] = pd.NaT

    uc = uc.dropna(subset=["user_id", "competence_id", "probability"])
    uc["probability"] = uc["probability"].clip(0, 1)

    uc = uc.sort_values(
        ["user_id", "competence_id", "evidence_count", "assessed_at"],
        ascending=[True, True, True, True]
    )

    uc_latest = (
        uc
        .groupby(["user_id", "competence_id"], as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )

    uc_latest["mastery"] = uc_latest["probability"]
    uc_latest["need"] = 1 - uc_latest["mastery"]

    return uc_latest

Актуальная таблица UserCompetence

In [12]:
uc_latest = prepare_user_competences(user_comp_df)

print("Было строк:", user_comp_df.shape)
print("Стало строк:", uc_latest.shape)
print("Users:", uc_latest["user_id"].nunique())
print("Competences:", uc_latest["competence_id"].nunique())

display(uc_latest.head())

Было строк: (7850, 11)
Стало строк: (1006, 13)
Users: 31
Competences: 94


,user_number,competence_id,competence_code,competence_name,probability,assessment_source,evidence_count,assessed_at,valid_from,valid_to,user_id,mastery,need
0,1,078423e2-9dcd-4e8e-9887-702ee08b0153,6.2.2,Умеет приводить подобные слагаемые и упрощать ...,0.956498,calculated,6,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,1,0.956498,0.043502
1,1,1024e3ec-6786-4059-84fe-3ae001527444,7.0.0,Вычисления и преобразования,0.035088,calculated,2,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,1,0.035088,0.964912
2,1,118d8585-a00f-46f7-be45-0a39fc33f401,6.16.1,Умеет представлять числа и дроби в виде степен...,0.115756,calculated,2,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,1,0.115756,0.884244
3,1,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4.1.4,"Умеет вычислять значение отношения двух чисел,...",0.629160,calculated,4,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,1,0.629160,0.370840
4,1,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,6.14.1,Умеет интерпретировать корень нечётной степени...,0.035088,calculated,1,2026-03-15 18:13:54.078542,2026-03-15 18:13:54.078542,NaN,1,0.035088,0.964912


In [13]:
def build_user_competence_trends(user_comp_df):
    uc = user_comp_df.copy()

    uc["user_id"] = uc["user_id"].astype(str)
    uc["competence_id"] = uc["competence_id"].astype(str)

    uc["probability"] = pd.to_numeric(uc["probability"], errors="coerce")
    uc["evidence_count"] = pd.to_numeric(uc["evidence_count"], errors="coerce").fillna(0)

    if "assessed_at" in uc.columns:
        uc["assessed_at"] = pd.to_datetime(uc["assessed_at"], errors="coerce")
    else:
        uc["assessed_at"] = pd.NaT

    uc = uc.dropna(subset=["user_id", "competence_id", "probability"])
    uc["probability"] = uc["probability"].clip(0, 1)

    rows = []

    for (user_id, competence_id), group in uc.groupby(["user_id", "competence_id"]):
        group = group.sort_values(
            ["evidence_count", "assessed_at"],
            ascending=[True, True]
        )

        probs = group["probability"].values
        evidences = group["evidence_count"].values

        n_points = len(group)

        first_prob = probs[0]
        last_prob = probs[-1]
        prob_delta = last_prob - first_prob

        if n_points >= 2:
            prev_prob = probs[-2]
            recent_delta = last_prob - prev_prob
        else:
            prev_prob = np.nan
            recent_delta = 0.0

        if n_points >= 2:
            x = np.arange(n_points)
            trend_slope = np.polyfit(x, probs, 1)[0]
        else:
            trend_slope = 0.0

        decline_score = max(0, -prob_delta)
        recent_decline_score = max(0, -recent_delta)

        if n_points >= 2 and abs(prob_delta) <= 0.03 and last_prob < 0.5:
            stagnation_score = 1.0
        else:
            stagnation_score = 0.0

        # trend_score высокий, если пользователь проседает или застрял низко.
        trend_score = (
            1.3 * decline_score
            + 0.7 * recent_decline_score
        )
        trend_score = min(1.0, max(0.0, trend_score))

        if n_points < 2:
            trend_label = "no_history"
        elif prob_delta > 0.03:
            trend_label = "improving"
        elif prob_delta < -0.03:
            trend_label = "declining"
        elif last_prob < 0.5:
            trend_label = "stuck_low"
        else:
            trend_label = "stable"

        rows.append({
            "user_id": user_id,
            "competence_id": competence_id,
            "trend_points": n_points,
            "first_probability": first_prob,
            "last_probability": last_prob,
            "probability_delta": prob_delta,
            "recent_probability_delta": recent_delta,
            "trend_slope": trend_slope,
            "trend_score": trend_score,
            "trend_label": trend_label
        })

    return pd.DataFrame(rows)

In [14]:
trend_df = build_user_competence_trends(user_comp_df)

print("trend_df:", trend_df.shape)
display(trend_df.head())
print(trend_df["trend_label"].value_counts())

trend_df: (1006, 10)


,user_id,competence_id,trend_points,first_probability,last_probability,probability_delta,recent_probability_delta,trend_slope,trend_score,trend_label
0,1,078423e2-9dcd-4e8e-9887-702ee08b0153,3,0.115756,0.956498,0.840743,0.327338,0.420371,0.0,improving
1,1,1024e3ec-6786-4059-84fe-3ae001527444,2,0.001345,0.035088,0.033743,0.033743,0.033743,0.0,improving
2,1,118d8585-a00f-46f7-be45-0a39fc33f401,1,0.115756,0.115756,0.000000,0.000000,0.000000,0.0,no_history
3,1,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4,0.035088,0.629160,0.594072,0.308844,0.198678,0.0,improving
4,1,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,1,0.035088,0.035088,0.000000,0.000000,0.000000,0.0,no_history


trend_label
improving     470
no_history    422
stuck_low      81
declining      28
stable          5
Name: count, dtype: int64


Подготовка опыта пользователя по задачам

* сколько задач решал;
* сколько правильно;
* по каким компетенциям были попытки;
* какой средний score;
* когда была последняя попытка.

In [15]:
def prepare_task_experience(task_submissions, task_competences):
    ts = task_submissions.copy()
    tc = task_competences.copy()

    if ts.empty or tc.empty:
        return pd.DataFrame(columns=[
            "user_id", "competence_id", "task_attempts", "task_correct_attempts",
            "task_correct_rate", "task_avg_score", "last_task_time"
        ])

    ts["user_id"] = ts["user_id"].astype(str)
    ts["task_id"] = ts["task_id"].astype(str)

    tc["task_id"] = tc["task_id"].astype(str)
    tc["competence_id"] = tc["competence_id"].astype(str)

    if "is_correct" in ts.columns:
        ts["is_correct_num"] = ts["is_correct"].astype(bool).astype(int)
    else:
        ts["is_correct_num"] = 0

    ts["score"] = pd.to_numeric(ts.get("score", 0), errors="coerce").fillna(0)
    ts["max_score"] = pd.to_numeric(ts.get("max_score", 1), errors="coerce").replace(0, np.nan)

    ts["score_share"] = (
        ts["score"] / ts["max_score"]
    ).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

    if "submission_time" in ts.columns:
        ts["submission_time"] = pd.to_datetime(ts["submission_time"], errors="coerce", utc=True)
    else:
        ts["submission_time"] = pd.NaT

    tc_small = tc[["task_id", "competence_id"]].drop_duplicates()

    merged = ts.merge(tc_small, on="task_id", how="inner")

    exp = (
        merged
        .groupby(["user_id", "competence_id"], as_index=False)
        .agg(
            task_attempts=("task_id", "count"),
            task_correct_attempts=("is_correct_num", "sum"),
            task_correct_rate=("is_correct_num", "mean"),
            task_avg_score=("score_share", "mean"),
            last_task_time=("submission_time", "max")
        )
    )

    return exp

In [16]:
task_exp = prepare_task_experience(task_sub_df, task_comp_df)
print("task_exp:", task_exp.shape)
display(task_exp.head())

task_exp: (1264, 7)


,user_id,competence_id,task_attempts,task_correct_attempts,task_correct_rate,task_avg_score,last_task_time
0,1,078423e2-9dcd-4e8e-9887-702ee08b0153,6,6,1.0,1.0,2026-03-16 18:27:29.926178+00:00
1,1,1024e3ec-6786-4059-84fe-3ae001527444,2,1,0.5,0.5,2026-03-16 18:27:29.926178+00:00
2,1,118d8585-a00f-46f7-be45-0a39fc33f401,2,2,1.0,1.0,2026-03-16 18:27:29.926178+00:00
3,1,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4,4,1.0,1.0,2026-03-16 18:27:29.926178+00:00
4,1,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,1,1,1.0,1.0,2026-03-15 18:13:52.861923+00:00


Общий профиль пользователя

По ней можно понять:

* сколько всего задач решил пользователь;
* какой у него correct rate;
* сколько было практик;
* сколько было КИМов;
* сколько уникальных задач и микромодулей он трогал.

In [17]:
def prepare_user_summary(task_submissions, practice_submissions):
    frames = []

    if not task_submissions.empty:
        ts = task_submissions.copy()
        ts["user_id"] = ts["user_id"].astype(str)

        if "is_correct" in ts.columns:
            ts["is_correct_num"] = ts["is_correct"].astype(bool).astype(int)
        else:
            ts["is_correct_num"] = 0

        if "submission_time" in ts.columns:
            ts["submission_time"] = pd.to_datetime(ts["submission_time"], errors="coerce", utc=True)
        else:
            ts["submission_time"] = pd.NaT

        task_summary = (
            ts
            .groupby("user_id", as_index=False)
            .agg(
                solved_tasks=("task_id", "count"),
                unique_tasks=("task_id", "nunique"),
                unique_nodes=("node_id", "nunique"),
                task_correct_rate=("is_correct_num", "mean"),
                first_task_time=("submission_time", "min"),
                last_task_time=("submission_time", "max")
            )
        )

        frames.append(task_summary)

    if not practice_submissions.empty:
        pr = practice_submissions.copy()
        pr["user_id"] = pr["user_id"].astype(str)

        if "submission_time" in pr.columns:
            pr["submission_time"] = pd.to_datetime(pr["submission_time"], errors="coerce", utc=True)
        else:
            pr["submission_time"] = pd.NaT

        if "type" in pr.columns:
            pr["type"] = pr["type"].fillna("UNKNOWN")
        else:
            pr["type"] = "UNKNOWN"

        practice_summary = (
            pr
            .groupby("user_id", as_index=False)
            .agg(
                total_practices=("practice_submission_id", "count"),
                kim_count=("type", lambda x: (x == "KIM").sum()),
                block_count=("type", lambda x: (x != "KIM").sum()),
                avg_test_score=("test_score", "mean"),
                avg_total_score=("total_score", "mean"),
                first_practice_time=("submission_time", "min"),
                last_practice_time=("submission_time", "max")
            )
        )

        frames.append(practice_summary)

    if not frames:
        return pd.DataFrame(columns=["user_id"])

    out = frames[0]
    for frame in frames[1:]:
        out = out.merge(frame, on="user_id", how="outer")

    return out

In [18]:
user_summary = prepare_user_summary(task_sub_df, practice_df)
print("user_summary:", user_summary.shape)
display(user_summary.head())

user_summary: (39, 14)


,user_id,solved_tasks,unique_tasks,unique_nodes,task_correct_rate,first_task_time,last_task_time,total_practices,kim_count,block_count,avg_test_score,avg_total_score,first_practice_time,last_practice_time
0,1,64.0,58.0,17.0,0.953125,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00,9,4,5,38.555556,6.777778,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00
1,10,16.0,16.0,16.0,0.750000,2026-02-16 19:19:43.923400+00:00,2026-02-16 19:19:43.923400+00:00,1,1,0,90.000000,22.000000,2026-02-16 19:19:43.923400+00:00,2026-02-16 19:19:43.923400+00:00
2,11,192.0,159.0,52.0,0.401042,2026-02-04 22:23:20.665872+00:00,2026-03-23 12:36:14.238600+00:00,50,43,7,9.900000,1.740000,2025-12-30 10:41:20.433434+00:00,2026-05-13 19:33:13.226460+00:00
3,12,4.0,4.0,4.0,0.250000,NaT,NaT,3,3,0,2.000000,0.333333,2026-01-19 16:10:32.417875+00:00,2026-01-19 16:32:20.242732+00:00
4,13,474.0,203.0,44.0,0.860759,2026-02-18 19:41:18.709332+00:00,2026-05-08 15:34:47.662449+00:00,71,16,55,30.000000,5.746479,2026-02-18 19:41:18.709332+00:00,2026-05-10 12:56:28.996512+00:00


Текстовое описание компетенции

In [19]:
def build_competence_catalog(uc_latest, micro_df, task_comp_df):
    base_frames = []

    for df in [uc_latest, micro_df, task_comp_df]:
        cols = [
            col for col in ["competence_id", "competence_code", "competence_name"]
            if col in df.columns
        ]

        if "competence_id" in cols:
            base_frames.append(df[cols].drop_duplicates())

    if not base_frames:
        raise ValueError("Не удалось собрать каталог компетенций: нет competence_id")

    catalog = (
        pd.concat(base_frames, ignore_index=True)
        .drop_duplicates(subset=["competence_id"])
        .reset_index(drop=True)
    )

    catalog["competence_id"] = catalog["competence_id"].astype(str)

    if "competence_code" not in catalog.columns:
        catalog["competence_code"] = ""
    if "competence_name" not in catalog.columns:
        catalog["competence_name"] = ""

    catalog["competence_code"] = catalog["competence_code"].fillna("").astype(str)
    catalog["competence_name"] = catalog["competence_name"].fillna("").astype(str)

    catalog["competence_text"] = catalog["competence_name"]

    empty_mask = catalog["competence_text"].str.strip() == ""
    catalog.loc[empty_mask, "competence_text"] = catalog.loc[empty_mask, "competence_code"]

    empty_mask = catalog["competence_text"].str.strip() == ""
    catalog.loc[empty_mask, "competence_text"] = catalog.loc[empty_mask, "competence_id"]

    return catalog

In [20]:
comp_catalog = build_competence_catalog(uc_latest, micro_df, task_comp_df)

print("comp_catalog:", comp_catalog.shape)
display(comp_catalog.head())
print("Пустых competence_text:", (comp_catalog["competence_text"].str.len() == 0).sum())

comp_catalog: (327, 4)


,competence_id,competence_code,competence_name,competence_text
0,078423e2-9dcd-4e8e-9887-702ee08b0153,6.2.2,Умеет приводить подобные слагаемые и упрощать ...,Умеет приводить подобные слагаемые и упрощать ...
1,1024e3ec-6786-4059-84fe-3ae001527444,7.0.0,Вычисления и преобразования,Вычисления и преобразования
2,118d8585-a00f-46f7-be45-0a39fc33f401,6.16.1,Умеет представлять числа и дроби в виде степен...,Умеет представлять числа и дроби в виде степен...
3,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4.1.4,"Умеет вычислять значение отношения двух чисел,...","Умеет вычислять значение отношения двух чисел,..."
4,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,6.14.1,Умеет интерпретировать корень нечётной степени...,Умеет интерпретировать корень нечётной степени...


Пустых competence_text: 0


Объединяем UserCompetence и опыт пользователя

* probability — насколько пользователь владеет компетенцией;
* need — насколько её нужно подтянуть;
* evidence_count — сколько было касаний;
* task_attempts — сколько задач по этой компетенции решал пользователь;
* als_value — итоговый сигнал для ALS.

Формула для ALS:

als_value = (0.05 + need) × (1 + log(1 + evidence_count + task_attempts))

In [21]:
def build_model_table(uc_latest, task_exp, trend_df):
    df = uc_latest.copy()

    df = df.merge(task_exp, on=["user_id", "competence_id"], how="left")
    df = df.merge(trend_df, on=["user_id", "competence_id"], how="left")

    for col in ["task_attempts", "task_correct_attempts", "task_correct_rate", "task_avg_score"]:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    trend_numeric_cols = [
        "trend_points", "first_probability", "last_probability",
        "probability_delta", "recent_probability_delta",
        "trend_slope", "trend_score"
    ]

    for col in trend_numeric_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    if "trend_label" not in df.columns:
        df["trend_label"] = "no_history"
    df["trend_label"] = df["trend_label"].fillna("no_history")

    df["total_experience"] = df["evidence_count"].fillna(0) + df["task_attempts"].fillna(0)
    df["experience_weight"] = np.log1p(df["total_experience"])

    df["need"] = (1 - df["probability"].clip(0, 1)).clip(0, 1)

    base_als_value = (
        (0.05 + df["need"])
        * (1 + df["experience_weight"])
    )

    df["als_value"] = base_als_value * (1 + 0.25 * df["trend_score"])
    df["als_value"] = pd.to_numeric(df["als_value"], errors="coerce").fillna(0)

    df = df[df["als_value"] > 0].copy()

    return df

In [22]:
model_df = build_model_table(uc_latest, task_exp, trend_df)

print("model_df:", model_df.shape)
print("Users:", model_df["user_id"].nunique())
print("Competences:", model_df["competence_id"].nunique())

display(model_df[[
    "user_id", "competence_id", "competence_name", "probability", "need",
    "evidence_count", "task_attempts", "trend_score", "trend_label", "als_value"
]].head())

model_df: (1006, 29)
Users: 31
Competences: 94


,user_id,competence_id,competence_name,probability,need,evidence_count,task_attempts,trend_score,trend_label,als_value
0,1,078423e2-9dcd-4e8e-9887-702ee08b0153,Умеет приводить подобные слагаемые и упрощать ...,0.956498,0.043502,6,6.0,0.0,improving,0.333328
1,1,1024e3ec-6786-4059-84fe-3ae001527444,Вычисления и преобразования,0.035088,0.964912,2,2.0,0.0,improving,2.648351
2,1,118d8585-a00f-46f7-be45-0a39fc33f401,Умеет представлять числа и дроби в виде степен...,0.115756,0.884244,2,2.0,0.0,no_history,2.437853
3,1,2e7f87eb-5a14-4b20-99f4-7efc507e829b,"Умеет вычислять значение отношения двух чисел,...",0.629160,0.370840,4,4.0,0.0,improving,1.345520
4,1,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,Умеет интерпретировать корень нечётной степени...,0.035088,0.964912,1,1.0,0.0,no_history,2.129907


Строим матрицу user × competence для als

In [23]:
def build_sparse_matrix(df, value_col):
    users = df["user_id"].astype("category")
    comps = df["competence_id"].astype("category")

    user_index = users.cat.categories.astype(str)
    comp_index = comps.cat.categories.astype(str)

    rows = users.cat.codes
    cols = comps.cat.codes
    values = df[value_col].astype(float).values

    matrix = csr_matrix(
        (values, (rows, cols)),
        shape=(len(user_index), len(comp_index))
    )

    return matrix, user_index, comp_index

In [24]:
als_matrix, user_index, comp_index = build_sparse_matrix(model_df, "als_value")

print("als_matrix:", als_matrix.shape)
print("non-zero:", als_matrix.nnz)

sparsity = 1 - als_matrix.nnz / (als_matrix.shape[0] * als_matrix.shape[1])
print(f"sparsity: {sparsity:.2%}")

als_matrix: (31, 94)
non-zero: 1006
sparsity: 65.48%


Обучение ALS

ALS отвечает за опыт похожих пользователей.

In [25]:
ALS_FACTORS = 32
ALS_REGULARIZATION = 0.10
ALS_ITERATIONS = 50
ALS_ALPHA = 20.0

In [26]:
def train_als(matrix):
    model = AlternatingLeastSquares(
        factors=ALS_FACTORS,
        regularization=ALS_REGULARIZATION,
        iterations=ALS_ITERATIONS,
        random_state=RANDOM_STATE
    )

    model.fit((matrix * ALS_ALPHA).tocsr())
    return model

In [27]:
als_model = train_als(als_matrix)

print("ALS обучен")
print("user_factors:", als_model.user_factors.shape)
print("item_factors:", als_model.item_factors.shape)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

ALS обучен
user_factors: (31, 32)
item_factors: (94, 32)


Текстовые векторы компетенций

Теперь добавляем вторую часть рекомендательной системы — текстовую.

Каждая компетенция превращается в TF-IDF-вектор по её текстовому описанию.

In [28]:
item_catalog = pd.DataFrame({
    "competence_id": comp_index.astype(str)
})

item_catalog = item_catalog.merge(comp_catalog, on="competence_id", how="left")

item_catalog["competence_name"] = item_catalog["competence_name"].fillna("")
item_catalog["competence_code"] = item_catalog["competence_code"].fillna("")
item_catalog["competence_text"] = item_catalog["competence_name"]

empty_mask = item_catalog["competence_text"].str.strip() == ""
item_catalog.loc[empty_mask, "competence_text"] = item_catalog.loc[empty_mask, "competence_code"]

empty_mask = item_catalog["competence_text"].str.strip() == ""
item_catalog.loc[empty_mask, "competence_text"] = item_catalog.loc[empty_mask, "competence_id"]

In [29]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    lowercase=True
)

item_text_matrix = vectorizer.fit_transform(item_catalog["competence_text"])
item_text_matrix = normalize(item_text_matrix)

print("item_text_matrix:", item_text_matrix.shape)
print("Пример текста компетенции:")
print(item_catalog.loc[0, "competence_text"])

item_text_matrix: (94, 1048)
Пример текста компетенции:
Умеет находить производную функции вида ln(ax + b)^n, используя правило: (ln(ax + b)^n)' = n·a / (ax + b)


Вспомогательные словари

Нужно быстро получать:

* индекс пользователя;
* индекс компетенции;
* текущее состояние user × competence;
* историю пользователя.

In [30]:
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_index.astype(str))}
comp_to_idx = {comp_id: idx for idx, comp_id in enumerate(comp_index.astype(str))}

In [31]:
user_comp_state = (
    model_df
    .set_index(["user_id", "competence_id"])
    [[
        "probability", "need", "evidence_count", "task_attempts",
        "total_experience", "experience_weight",
        "trend_score", "trend_label", "probability_delta",
        "recent_probability_delta", "trend_slope"
    ]]
    .to_dict("index")
)

In [32]:
def minmax(x):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    mn = x.min()
    mx = x.max()

    if mx - mn < 1e-12:
        return np.zeros_like(x)

    return (x - mn) / (mx - mn)

In [33]:
def get_user_history(user_id):
    return model_df[model_df["user_id"].astype(str) == str(user_id)].copy()

Текстовый профиль слабых зон пользователя

Для каждого пользователя строим текстовый профиль его слабых мест.

* probability низкая
* need высокий
* есть evidence_count или task_attempts

In [34]:
def build_user_weak_text_profile(user_id):
    hist = get_user_history(user_id)

    if hist.empty:
        return None

    hist["comp_idx"] = hist["competence_id"].map(comp_to_idx)
    hist = hist.dropna(subset=["comp_idx"]).copy()
    hist["comp_idx"] = hist["comp_idx"].astype(int)

    weights = hist["need"].values * (1 + hist["experience_weight"].values)

    if np.isclose(weights.sum(), 0):
        weights = 1 + hist["experience_weight"].values

    profile = weights.reshape(1, -1) @ item_text_matrix[hist["comp_idx"].values]
    profile = normalize(profile)

    return profile

Гибридная рекомендация компетенций

Он использует 4 сигнала:

* ALS-score — опыт похожих пользователей;
* text-score — похожесть на слабые зоны пользователя;
* need-score — насколько компетенция плохо освоена;
* experience-score — насколько по этой компетенции уже есть опыт.


final_score =
    0.45 × ALS_score + 0.25 × text_score + 0.25 × need_score + 0.05 × experience_score

В прошлый раз я исключал все известные компетенции. Здесь так делать нельзя. Если пользователь уже видел компетенцию, но плохо её знает, её как раз надо рекомендовать.

Поэтому исключаем только те компетенции, которые уже хорошо освоены.

In [35]:
def recommend_competences_hybrid(
    user_id,
    top_n=30,
    w_als=0.30,
    w_text=0.20,
    w_need=0.25,
    w_experience=0.10,
    w_trend=0.15,
    mastery_threshold=0.75,
    min_evidence_for_mastery=3,
    keep_seen_weak=True,
    min_evidence_filter=None
):
    user_id = str(user_id)
    n_items = len(comp_index)

    # 1. ALS-score
    if user_id in user_to_idx:
        user_idx = user_to_idx[user_id]
        user_vec = als_model.user_factors[user_idx]
        als_scores = als_model.item_factors @ user_vec
    else:
        als_scores = np.zeros(n_items)

    # 2. Text-score
    user_profile = build_user_weak_text_profile(user_id)

    if user_profile is not None:
        text_scores_raw = item_text_matrix @ user_profile.T

        if hasattr(text_scores_raw, "toarray"):
            text_scores = text_scores_raw.toarray().ravel()
        else:
            text_scores = np.asarray(text_scores_raw).ravel()
    else:
        text_scores = np.zeros(n_items)

    # 3. Need, experience, trend
    need_scores = np.zeros(n_items)
    experience_scores = np.zeros(n_items)
    trend_scores = np.zeros(n_items)

    current_mastery = np.full(n_items, np.nan)
    current_evidence = np.zeros(n_items)
    current_task_attempts = np.zeros(n_items)

    trend_labels = np.array(["no_history"] * n_items, dtype=object)

    for comp_id, idx in comp_to_idx.items():
        state = user_comp_state.get((user_id, comp_id))

        if state is None:
            need_scores[idx] = 0.35
            experience_scores[idx] = 0.0
            trend_scores[idx] = 0.0
            trend_labels[idx] = "no_history"
        else:
            current_mastery[idx] = state["probability"]
            current_evidence[idx] = state["evidence_count"]
            current_task_attempts[idx] = state["task_attempts"]

            need_scores[idx] = state["need"]
            experience_scores[idx] = state["experience_weight"]
            trend_scores[idx] = state.get("trend_score", 0.0)
            trend_labels[idx] = state.get("trend_label", "no_history")

    final_scores = (
        w_als * minmax(als_scores)
        + w_text * minmax(text_scores)
        + w_need * minmax(need_scores)
        + w_experience * minmax(experience_scores)
        + w_trend * minmax(trend_scores)
    )

    rec = pd.DataFrame({
        "user_id": user_id,
        "competence_id": comp_index.astype(str),
        "competence_code": item_catalog["competence_code"].values,
        "competence_name": item_catalog["competence_name"].values,

        "als_score_raw": als_scores,
        "text_score_raw": text_scores,
        "need_score_raw": need_scores,
        "experience_score_raw": experience_scores,
        "trend_score_raw": trend_scores,

        "als_score": minmax(als_scores),
        "text_score": minmax(text_scores),
        "need_score": minmax(need_scores),
        "experience_score": minmax(experience_scores),
        "trend_score": minmax(trend_scores),

        "final_score": final_scores,

        "current_mastery": current_mastery,
        "current_evidence": current_evidence,
        "current_task_attempts": current_task_attempts,
        "trend_label": trend_labels
    })

    # Необязательный тестовый фильтр надёжности.
    # Для baseline лучше оставить min_evidence_filter=None.
    if min_evidence_filter is not None:
        rec = rec[
            (rec["current_evidence"] >= min_evidence_filter)
            | (rec["current_mastery"].isna())
        ].copy()

    # Убираем только хорошо освоенные компетенции.
    mastered_mask = (
        rec["current_mastery"].notna()
        & (rec["current_mastery"] >= mastery_threshold)
        & (rec["current_evidence"] >= min_evidence_for_mastery)
    )

    if keep_seen_weak:
        rec = rec[~mastered_mask].copy()
    else:
        seen_mask = rec["current_mastery"].notna()
        rec = rec[~seen_mask & ~mastered_mask].copy()

    conditions = [
        rec["trend_score"] >= 0.70,
        rec["need_score"] >= 0.70,
        rec["text_score"] >= 0.70,
        rec["als_score"] >= 0.70
    ]

    choices = [
        "негативная или застывшая динамика UserCompetence",
        "низкое владение по UserCompetence",
        "похожа на слабые зоны пользователя по тексту",
        "рекомендуется ALS по опыту похожих пользователей"
    ]

    rec["main_reason"] = np.select(
        conditions,
        choices,
        default="смешанный гибридный сигнал"
    )

    rec = rec.sort_values("final_score", ascending=False).reset_index(drop=True)
    rec["rank"] = np.arange(1, len(rec) + 1)

    return rec.head(top_n)

Пример рекомендации компетенций

In [36]:
example_user = str(user_index[0])

example_comp_rec = recommend_competences_hybrid(
    user_id=example_user,
    top_n=30,
    mastery_threshold=0.75,
    keep_seen_weak=True,
    min_evidence_filter=None
)

print("example_user:", example_user)

display(example_comp_rec[[
    "rank", "competence_code", "competence_name",
    "final_score", "als_score", "text_score", "need_score",
    "experience_score", "trend_score", "trend_label",
    "current_mastery", "current_evidence", "current_task_attempts",
    "main_reason"
]])

example_user: 1


,rank,competence_code,competence_name,final_score,als_score,text_score,need_score,experience_score,trend_score,trend_label,current_mastery,current_evidence,current_task_attempts,main_reason
0,1,9.2.1.,Умеет интерпретировать условие «значение выраж...,0.766255,0.986714,1.000000,0.916399,0.411408,0.0,improving,0.115756,2.0,2.0,низкое владение по UserCompetence
1,2,9.4.4.,Умеет интерпретировать условие «значение выраж...,0.753889,0.998233,0.920895,0.916399,0.411408,0.0,improving,0.115756,2.0,2.0,низкое владение по UserCompetence
2,3,4.2.2,Умеет находить число благоприятных исходов чер...,0.737182,0.996643,0.800529,1.000000,0.280830,0.0,no_history,0.035088,1.0,1.0,низкое владение по UserCompetence
3,4,4.5.2,Умеет находить число благоприятных исходов чер...,0.726908,0.994395,0.752534,1.000000,0.280830,0.0,no_history,0.035088,1.0,1.0,низкое владение по UserCompetence
4,5,4.1.2,Умеет определять число благоприятных исходов m...,0.724127,0.994797,0.999231,0.704399,0.497418,0.0,improving,0.320316,3.0,3.0,низкое владение по UserCompetence
5,6,6.1.3,Умеет решать уравнение вида a/b·x = c/d через ...,0.723402,0.993314,0.736624,1.000000,0.280830,0.0,no_history,0.035088,1.0,1.0,низкое владение по UserCompetence
6,7,8.4.3,Умеет решать квадратное уравнение,0.721548,0.987740,0.735715,1.000000,0.280830,0.0,no_history,0.035088,1.0,1.0,низкое владение по UserCompetence
7,8,6.15.1,Умеет представлять число в виде степени с нужн...,0.719727,0.986073,0.729109,1.000000,0.280830,0.0,no_history,0.035088,1.0,1.0,низкое владение по UserCompetence
8,9,6.1.2,Умеет решать уравнение вида a/b·x = c/d с испо...,0.719401,0.986300,0.727140,1.000000,0.280830,0.0,no_history,0.035088,1.0,1.0,низкое владение по UserCompetence
9,10,6.16.1,Умеет представлять числа и дроби в виде степен...,0.713897,0.993539,0.727975,0.916399,0.411408,0.0,no_history,0.115756,2.0,2.0,низкое владение по UserCompetence


Рекомендации компетенций для всех пользователей

In [37]:
all_comp_recs = []

for user_id in tqdm(user_index.astype(str), desc="Hybrid competence recommendations"):
    rec = recommend_competences_hybrid(
        user_id=user_id,
        top_n=30,
        mastery_threshold=0.75,
        keep_seen_weak=True,
        min_evidence_filter=None
    )

    all_comp_recs.append(rec)

hybrid_competence_rec = pd.concat(all_comp_recs, ignore_index=True)

display(hybrid_competence_rec.head())

hybrid_competence_rec.to_csv(
    OUTPUT_DIR / "hybrid_competence_recommendations_with_trend.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Сохранено:", OUTPUT_DIR / "hybrid_competence_recommendations_with_trend.csv")

Hybrid competence recommendations: 100%|██████████| 31/31 [00:00<00:00, 103.47it/s]


,user_id,competence_id,competence_code,competence_name,als_score_raw,text_score_raw,need_score_raw,experience_score_raw,trend_score_raw,als_score,...,need_score,experience_score,trend_score,final_score,current_mastery,current_evidence,current_task_attempts,trend_label,main_reason,rank
0,1,6aafe9b9-9d86-442a-9054-0372b4ff08d6,9.2.1.,Умеет интерпретировать условие «значение выраж...,0.993551,0.370655,0.884244,1.609438,0.0,0.986714,...,0.916399,0.411408,0.0,0.766255,0.115756,2.0,2.0,improving,низкое владение по UserCompetence,1
1,1,b4cd88be-c890-4d73-9c8a-2071f67c2af2,9.4.4.,Умеет интерпретировать условие «значение выраж...,1.005828,0.341335,0.884244,1.609438,0.0,0.998233,...,0.916399,0.411408,0.0,0.753889,0.115756,2.0,2.0,improving,низкое владение по UserCompetence,2
2,1,30b793a9-7275-40f1-9b91-6aed9314431d,4.2.2,Умеет находить число благоприятных исходов чер...,1.004133,0.296721,0.964912,1.098612,0.0,0.996643,...,1.000000,0.280830,0.0,0.737182,0.035088,1.0,1.0,no_history,низкое владение по UserCompetence,3
3,1,6c4e8a36-4884-40cf-848a-5f7174444d79,4.5.2,Умеет находить число благоприятных исходов чер...,1.001737,0.278931,0.964912,1.098612,0.0,0.994395,...,1.000000,0.280830,0.0,0.726908,0.035088,1.0,1.0,no_history,низкое владение по UserCompetence,4
4,1,5067001a-a115-4624-823b-4de68176f569,4.1.2,Умеет определять число благоприятных исходов m...,1.002166,0.370371,0.679684,1.945910,0.0,0.994797,...,0.704399,0.497418,0.0,0.724127,0.320316,3.0,3.0,improving,низкое владение по UserCompetence,5


Сохранено: hybrid_recommender_output_with_trend/hybrid_competence_recommendations_with_trend.csv


Переход от компетенций к микромодулям

coverage_score — какую долю компетенций микромодуля покрывает рекомендация;


jaccard_score — пересечение рекомендованных компетенций и компетенций микромодуля;


weighted_coverage_score — то же самое, но с учётом final_score компетенций.

In [38]:
USE_ONLY_MAIN_MICROMODULES = False

In [39]:
def prepare_micro_source(micro_df):
    m = micro_df.copy()

    for col in ["node_id", "node_name", "competence_id", "course_description", "task_number"]:
        if col in m.columns:
            m[col] = m[col].astype(str)

    if USE_ONLY_MAIN_MICROMODULES and "importance" in m.columns:
        filtered = m[
            pd.to_numeric(m["importance"], errors="coerce").fillna(1.0) >= 1.0
        ].copy()

        if not filtered.empty:
            m = filtered

    return m

In [40]:
micro_source = prepare_micro_source(micro_df)

micro_comp_dict = (
    micro_source
    .groupby("node_id")["competence_id"]
    .apply(lambda x: set(x.astype(str)))
    .to_dict()
)

micro_meta = (
    micro_source
    .groupby("node_id", as_index=False)
    .agg(
        node_name=("node_name", "first"),
        task_number=("task_number", "first"),
        course_description=("course_description", "first"),
        micro_competences_count=("competence_id", "nunique")
    )
)

micro_meta_dict = micro_meta.set_index("node_id").to_dict("index")

comp_name_dict = (
    comp_catalog[["competence_id", "competence_name"]]
    .drop_duplicates("competence_id")
    .set_index("competence_id")["competence_name"]
    .to_dict()
)

print("Микромодулей:", len(micro_comp_dict))
display(micro_meta.head())

Микромодулей: 277


,node_id,node_name,task_number,course_description,micro_competences_count
0,0226f63b-043e-43c1-9db1-5157c951a514,19. Нахождение экстремумы функции с помощью пр...,12,Значения функций,6
1,02527317-f5cd-4d6e-84d0-3a55505645d5,22. Классическая вероятность в условном простр...,4,Теория Вероятностей,7
2,033c8dfe-1be1-45e0-8de8-509208836353,1. Вероятность выбора яйца из двух хозяйств с ...,5,Усложненная Теория Вероятностей,7
3,045b63b5-a649-4fe3-9df5-8afe52894f1f,25. Вероятность заданного числа бросков при из...,5,Усложненная Теория Вероятностей,9
4,05168121-2cbb-4706-9ff0-8f1828b1fb9c,3. Без производной - Определение экстремума ло...,12,Значения функций,4


In [41]:
def jaccard(a, b):
    a = set(a)
    b = set(b)

    union = a | b

    if len(union) == 0:
        return 0

    return len(a & b) / len(union)

Функция рекомендации микромодулей

In [42]:
def recommend_micro_modules(user_id, rec_df, top_comp=30, top_micro=10):
    user_id = str(user_id)

    user_rec = (
        rec_df[rec_df["user_id"].astype(str) == user_id]
        .sort_values("rank")
        .head(top_comp)
        .copy()
    )

    if user_rec.empty:
        return pd.DataFrame()

    rec_competences = user_rec["competence_id"].astype(str).tolist()
    rec_comp_set = set(rec_competences)

    rec_weight = dict(
        zip(
            user_rec["competence_id"].astype(str),
            user_rec["final_score"].astype(float)
        )
    )

    rows = []

    for node_id, node_comp_set in micro_comp_dict.items():
        matched = rec_comp_set & node_comp_set

        if len(matched) == 0:
            continue

        weighted_sum = sum(rec_weight.get(comp_id, 0) for comp_id in matched)

        coverage_score = len(matched) / max(len(node_comp_set), 1)
        weighted_coverage_score = weighted_sum / max(len(node_comp_set), 1)

        meta = micro_meta_dict.get(node_id, {})

        matched_names = [comp_name_dict.get(comp_id, comp_id) for comp_id in matched]

        rows.append({
            "user_id": user_id,
            "node_id": node_id,
            "node_name": meta.get("node_name", ""),
            "task_number": meta.get("task_number", ""),
            "course_description": meta.get("course_description", ""),
            "micro_competences_count": meta.get("micro_competences_count", len(node_comp_set)),
            "matched_competences_count": len(matched),
            "coverage_score": coverage_score,
            "jaccard_score": jaccard(rec_comp_set, node_comp_set),
            "weighted_coverage_score": weighted_coverage_score,
            "matched_competence_ids": " | ".join(sorted(matched)),
            "matched_competence_names": " | ".join(sorted(set(matched_names)))
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    result = result.sort_values(
        ["weighted_coverage_score", "coverage_score", "jaccard_score"],
        ascending=False
    ).reset_index(drop=True)

    result["rank"] = np.arange(1, len(result) + 1)

    return result.head(top_micro)

In [43]:
example_micro_rec = recommend_micro_modules(
    user_id=example_user,
    rec_df=hybrid_competence_rec,
    top_comp=30,
    top_micro=10
)

print("example_user:", example_user)

display(example_micro_rec[[
    "rank", "task_number", "node_name", "course_description",
    "weighted_coverage_score", "coverage_score", "jaccard_score",
    "matched_competences_count", "micro_competences_count",
    "matched_competence_names"
]])

example_user: 1


,rank,task_number,node_name,course_description,weighted_coverage_score,coverage_score,jaccard_score,matched_competences_count,micro_competences_count,matched_competence_names
0,1,7,Номер 7 ЕГЭ,Вычисления,0.704948,1.000,0.033333,1,1,Вычисления и преобразования
1,2,4,18. Вероятность “кто окажется k-м”: вычисление...,Теория Вероятностей,0.639955,1.000,0.200000,6,6,Умеет вычислять вероятность события по формуле...
2,3,4,5. Классическая вероятность при вычислении чис...,Теория Вероятностей,0.630895,1.000,0.166667,5,5,Умеет вычислять вероятность события по формуле...
3,4,4,2. Классическая вероятность: вычисление числа ...,Теория Вероятностей,0.621446,1.000,0.166667,5,5,Умеет вычислять вероятность события по формуле...
4,5,4,"12. Жеребьевка очередности: вероятность на ""фи...",Теория Вероятностей,0.620096,1.000,0.166667,5,5,Умеет вычислять вероятность события по формуле...
5,6,4,17. Игральные кости: классическая вероятность ...,Теория Вероятностей,0.618835,1.000,0.166667,5,5,Умеет вычислять вероятность события по формуле...
6,7,9,10. Расчет предельного времени/значения по фор...,Прикладные задачи,0.616483,0.875,0.225806,7,8,Умеет выбрать меньший/больший из корней уравне...
7,8,4,11. Классическая вероятность попадания в одну ...,Теория Вероятностей,0.606892,1.000,0.133333,4,4,Умеет вычислять вероятность события по формуле...
8,9,4,1. Классическое определение вероятности: равно...,Теория Вероятностей,0.606892,1.000,0.133333,4,4,Умеет вычислять вероятность события по формуле...
9,10,9,11. Расчет максимального времени в зоне связи ...,Прикладные задачи,0.605397,0.875,0.225806,7,8,Умеет выбрать меньший/больший из корней уравне...


In [44]:
all_micro_recs = []

for user_id in tqdm(user_index.astype(str), desc="Micro-module recommendations"):
    rec = recommend_micro_modules(
        user_id=user_id,
        rec_df=hybrid_competence_rec,
        top_comp=30,
        top_micro=10
    )

    if not rec.empty:
        all_micro_recs.append(rec)

if all_micro_recs:
    hybrid_micro_rec = pd.concat(all_micro_recs, ignore_index=True)
else:
    hybrid_micro_rec = pd.DataFrame()

display(hybrid_micro_rec.head())

Micro-module recommendations: 100%|██████████| 31/31 [00:00<00:00, 143.07it/s]


,user_id,node_id,node_name,task_number,course_description,micro_competences_count,matched_competences_count,coverage_score,jaccard_score,weighted_coverage_score,matched_competence_ids,matched_competence_names,rank
0,1,599e4cef-1b6a-4027-a8dc-5fad3647a149,Номер 7 ЕГЭ,7,Вычисления,1,1,1.0,0.033333,0.704948,1024e3ec-6786-4059-84fe-3ae001527444,Вычисления и преобразования,1
1,1,a5bbdc4d-a9ad-4a93-9ee2-37042e4139e0,18. Вероятность “кто окажется k-м”: вычисление...,4,Теория Вероятностей,6,6,1.0,0.200000,0.639955,2e7f87eb-5a14-4b20-99f4-7efc507e829b | 5067001...,Умеет вычислять вероятность события по формуле...,2
2,1,3d8e832a-5801-4316-b254-4d97a650fcff,5. Классическая вероятность при вычислении чис...,4,Теория Вероятностей,5,5,1.0,0.166667,0.630895,2e7f87eb-5a14-4b20-99f4-7efc507e829b | 5067001...,Умеет вычислять вероятность события по формуле...,3
3,1,60f0aedc-bb56-4d28-877d-83501edb4324,2. Классическая вероятность: вычисление числа ...,4,Теория Вероятностей,5,5,1.0,0.166667,0.621446,2e7f87eb-5a14-4b20-99f4-7efc507e829b | 30b793a...,Умеет вычислять вероятность события по формуле...,4
4,1,2387adbe-b043-4ee9-888c-2379a5c04b2c,"12. Жеребьевка очередности: вероятность на ""фи...",4,Теория Вероятностей,5,5,1.0,0.166667,0.620096,2e7f87eb-5a14-4b20-99f4-7efc507e829b | 5067001...,Умеет вычислять вероятность события по формуле...,5


In [45]:
hybrid_micro_rec.to_csv(
    OUTPUT_DIR / "hybrid_micro_module_recommendations_with_trend.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Сохранено:", OUTPUT_DIR / "hybrid_micro_module_recommendations_with_trend.csv")

Сохранено: hybrid_recommender_output_with_trend/hybrid_micro_module_recommendations_with_trend.csv


Сводка по пользователям

* сколько задач решил;
* какой correct rate;
* сколько компетенций оценено;
* средний уровень владения;
* средняя потребность;
* сколько всего evidence.

In [46]:
comp_user_agg = (
    model_df
    .groupby("user_id", as_index=False)
    .agg(
        assessed_competences=("competence_id", "nunique"),
        avg_mastery=("probability", "mean"),
        avg_need=("need", "mean"),
        total_evidence=("evidence_count", "sum"),
        avg_evidence=("evidence_count", "mean"),
        avg_trend_score=("trend_score", "mean"),
        declining_competences=("trend_label", lambda x: (x == "declining").sum()),
        stuck_low_competences=("trend_label", lambda x: (x == "stuck_low").sum()),
        improving_competences=("trend_label", lambda x: (x == "improving").sum())
    )
)

In [47]:
user_profile_summary = (
    pd.DataFrame({"user_id": user_index.astype(str)})
    .merge(user_summary, on="user_id", how="left")
    .merge(comp_user_agg, on="user_id", how="left")
)

display(user_profile_summary.head())

,user_id,solved_tasks,unique_tasks,unique_nodes,task_correct_rate,first_task_time,last_task_time,total_practices,kim_count,block_count,...,last_practice_time,assessed_competences,avg_mastery,avg_need,total_evidence,avg_evidence,avg_trend_score,declining_competences,stuck_low_competences,improving_competences
0,1,64.0,58.0,17.0,0.953125,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00,9,4,5,...,2026-03-22 09:40:01.786033+00:00,32,0.288507,0.711493,117,3.656250,0.000000,0,0,16
1,10,16.0,16.0,16.0,0.750000,2026-02-16 19:19:43.923400+00:00,2026-02-16 19:19:43.923400+00:00,1,1,0,...,2026-02-16 19:19:43.923400+00:00,25,0.032388,0.967612,25,1.000000,0.000000,0,0,0
2,11,192.0,159.0,52.0,0.401042,2026-02-04 22:23:20.665872+00:00,2026-03-23 12:36:14.238600+00:00,50,43,7,...,2026-05-13 19:33:13.226460+00:00,58,0.365170,0.634830,539,9.293103,0.022242,3,9,34
3,12,4.0,4.0,4.0,0.250000,NaT,NaT,3,3,0,...,2026-01-19 16:32:20.242732+00:00,2,0.018216,0.981784,2,1.000000,0.008655,0,2,0
4,13,474.0,203.0,44.0,0.860759,2026-02-18 19:41:18.709332+00:00,2026-05-08 15:34:47.662449+00:00,71,16,55,...,2026-05-10 12:56:28.996512+00:00,65,0.575824,0.424176,1310,20.153846,0.000652,0,3,53


In [48]:
user_profile_summary.to_csv(
    OUTPUT_DIR / "user_profile_summary_with_trend.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Сохранено:", OUTPUT_DIR / "user_profile_summary_with_trend.csv")

Сохранено: hybrid_recommender_output_with_trend/user_profile_summary_with_trend.csv


In [49]:
print("Компетентностные рекомендации:", hybrid_competence_rec.shape)
print("Микромодульные рекомендации:", hybrid_micro_rec.shape)
print("Профили пользователей:", user_profile_summary.shape)

print("\nПричины рекомендаций компетенций:")
print(hybrid_competence_rec["main_reason"].value_counts())

print("\nРаспределение trend_label в рекомендациях:")
print(hybrid_competence_rec["trend_label"].value_counts())

print("\nТоп курсов/разделов в микромодулях:")
if not hybrid_micro_rec.empty and "course_description" in hybrid_micro_rec.columns:
    print(hybrid_micro_rec["course_description"].value_counts().head(15))

Компетентностные рекомендации: (930, 21)
Микромодульные рекомендации: (310, 13)
Профили пользователей: (31, 23)

Причины рекомендаций компетенций:
main_reason
низкое владение по UserCompetence                   546
смешанный гибридный сигнал                          289
рекомендуется ALS по опыту похожих пользователей     39
негативная или застывшая динамика UserCompetence     39
похожа на слабые зоны пользователя по тексту         17
Name: count, dtype: int64

Распределение trend_label в рекомендациях:
trend_label
no_history    683
improving     144
stuck_low      75
declining      28
Name: count, dtype: int64

Топ курсов/разделов в микромодулях:
course_description
Значения функций                   75
Теория Вероятностей                59
Производная                        47
Планиметрия                        31
Простейшие уравнения               22
Усложненная Теория Вероятностей    17
Вычисления                         17
Стереометрия                       14
Текстовые задачи     